# 09 — Backtesting Dangers

**AFML Chapter 11** — López de Prado (2018)

Most published backtests are overfit.  The more strategies you test,
the more likely you are to find one that looks good *in-sample* but
fails out-of-sample.  This is selection bias.

This notebook demonstrates:
1. **Expected Maximum Sharpe** — the Sharpe you'd expect from N random strategies
2. **Deflated Sharpe Ratio (DSR)** — adjusts for selection bias + non-normality
3. **PBO via CPCV** — Probability of Backtest Overfitting from combinatorial paths

---

In [ ]:
from _setup import *
from afml.backtest_stats import (
    expected_max_sharpe,
    deflated_sharpe_ratio,
    probability_of_backtest_overfitting,
    sharpe_ratio,
    return_stats,
)
from afml.labeling import daily_volatility, make_events, triple_barrier_labels
from afml.sample_weights import sample_weight_by_return, normalize_weights
from afml.fracdiff import frac_diff_log
from afml.cross_validation import CombinatorialPurgedKFold
from validation.cv import PurgedKFold

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

## 1. Expected Maximum Sharpe vs number of trials

Even if all N strategies have true Sharpe = 0, the *best* observed
Sharpe among N trials grows as ~√(2·log(N)).  This is the selection
bias that must be corrected for.

In [ ]:
trials = [1, 5, 10, 25, 50, 100, 200, 500, 1000, 5000]
e_max = [expected_max_sharpe(n) for n in trials]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(trials, e_max, "o-", color=NAVY, lw=2, markersize=5)
ax.set_xscale("log")
ax.set_xlabel("Number of trials (log scale)")
ax.set_ylabel("E[max Sharpe]")
ax.set_title("Expected max Sharpe under the null (all strategies have SR=0)", fontweight="bold")

for n, e in zip([10, 100, 1000], [expected_max_sharpe(n) for n in [10, 100, 1000]]):
    ax.annotate(f"N={n:,}: SR={e:.2f}", (n, e), textcoords="offset points",
                xytext=(10, 5), fontsize=9)

fig.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "09_expected_max_sharpe.png"), dpi=150, bbox_inches="tight")
plt.show()

print("\nImplication: after testing 100 strategies, a Sharpe of"
      f" {expected_max_sharpe(100):.2f} is expected *by chance alone*.")

## 2. Deflated Sharpe Ratio on BTC strategy

In [ ]:
panel = load_daily_bars()
btc = panel[panel["symbol"] == "BTC-USD"].sort_values("ts").drop_duplicates("ts", keep="last")
close = btc.set_index("ts")["close"]

# Simple momentum strategy returns
rets = np.log(close / close.shift(1)).dropna()
signal = close.rolling(50).mean() > close.rolling(200).mean()
strat_rets = rets * signal.shift(1).reindex(rets.index).fillna(0)

stats = return_stats(strat_rets)
print(f"Strategy return stats:")
for k, v in stats.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

In [ ]:
# DSR for different numbers of trials
print(f"{'N trials':>10s} {'E[maxSR]':>10s} {'DSR':>10s} {'Verdict':>15s}")
print("-" * 50)

for n in [1, 5, 10, 50, 100, 500]:
    benchmark = expected_max_sharpe(n)
    dsr = deflated_sharpe_ratio(
        observed_sr=stats["sharpe"],
        sr_benchmark=benchmark,
        n_obs=stats["n_obs"],
        skewness=stats["skewness"],
        excess_kurtosis=stats["excess_kurtosis"],
    )
    verdict = "Significant" if dsr > 0.95 else "Marginal" if dsr > 0.5 else "Overfit"
    print(f"{n:>10d} {benchmark:>10.2f} {dsr:>10.1%} {verdict:>15s}")

## 3. PBO via CPCV

Using CPCV paths from notebook 04, we can estimate the Probability
of Backtest Overfitting: how often does the best in-sample path
underperform out-of-sample?

In [ ]:
# Prepare data
vol = daily_volatility(close, span=20)
events = make_events(close, vol, holding_periods=10)
tb = triple_barrier_labels(close, events, pt_sl=(2.0, 2.0))
tb["y"] = (tb["label"] == 1).astype(int)

feat = pd.DataFrame(index=tb.index)
feat["ret_1d"] = np.log(close / close.shift(1)).reindex(tb.index)
feat["ret_5d"] = np.log(close / close.shift(5)).reindex(tb.index)
feat["ret_21d"] = np.log(close / close.shift(21)).reindex(tb.index)
feat["vol_20d"] = vol.reindex(tb.index)
feat["fracdiff"] = frac_diff_log(close, d=0.3).reindex(tb.index)

df = feat.join(tb[["y", "t1"]]).dropna()
X = df[feat.columns].values
y_arr = df["y"].values
t1 = pd.Series(df["t1"].values, index=df.index)

# CPCV: collect IS and OOS scores for each path
cpcv = CombinatorialPurgedKFold(n_groups=6, n_test_groups=2, t1=t1, pct_embargo=0.01)

is_scores, oos_scores = [], []
for train_idx, test_idx in cpcv.split(X):
    rf = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42, n_jobs=-1)
    rf.fit(X[train_idx], y_arr[train_idx])

    is_acc = accuracy_score(y_arr[train_idx], rf.predict(X[train_idx]))
    oos_acc = accuracy_score(y_arr[test_idx], rf.predict(X[test_idx]))

    is_scores.append(is_acc)
    oos_scores.append(oos_acc)

is_scores = np.array(is_scores)
oos_scores = np.array(oos_scores)

print(f"CPCV paths: {len(is_scores)}")
print(f"IS accuracy:  {is_scores.mean():.1%} ± {is_scores.std():.1%}")
print(f"OOS accuracy: {oos_scores.mean():.1%} ± {oos_scores.std():.1%}")

In [ ]:
pbo = probability_of_backtest_overfitting(is_scores, oos_scores)

print(f"PBO Results:")
print(f"  P(Backtest Overfitting): {pbo['pbo']:.1%}")
print(f"  IS–OOS Rank Correlation: {pbo['rank_corr']:.3f}")
print(f"  Best IS path → OOS acc:  {pbo['best_is_oos']:.1%}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# IS vs OOS scatter
ax = axes[0]
ax.scatter(is_scores, oos_scores, color=NAVY, alpha=0.6, s=40, edgecolors="white")
best_idx = np.argmax(is_scores)
ax.scatter(is_scores[best_idx], oos_scores[best_idx], color=RED, s=100,
           zorder=5, edgecolors="black", label="Best IS path")
ax.plot([0.4, 0.7], [0.4, 0.7], color=GRAY, ls="--", lw=0.8)
ax.set_xlabel("In-sample accuracy")
ax.set_ylabel("Out-of-sample accuracy")
ax.set_title(f"IS vs OOS (rank corr: {pbo['rank_corr']:.2f})", fontweight="bold")
ax.legend(fontsize=9)

# IS and OOS histograms
ax = axes[1]
bins = np.linspace(min(is_scores.min(), oos_scores.min()) - 0.02,
                   max(is_scores.max(), oos_scores.max()) + 0.02, 12)
ax.hist(is_scores, bins=bins, alpha=0.5, color=NAVY, label="IS", edgecolor="white")
ax.hist(oos_scores, bins=bins, alpha=0.5, color=TEAL, label="OOS", edgecolor="white")
ax.axvline(0.5, color=RED, ls="--", lw=1, label="Random")
ax.set_xlabel("Accuracy")
ax.set_ylabel("Count")
ax.set_title(f"PBO = {pbo['pbo']:.1%}", fontweight="bold")
ax.legend(fontsize=9)

fig.suptitle("Probability of Backtest Overfitting (CPCV, 15 paths)",
             fontweight="bold", fontsize=13)
fig.tight_layout()
fig.savefig(str(ARTIFACTS_DIR / "09_pbo.png"), dpi=150, bbox_inches="tight")
plt.show()

## 4. Summary

**Key lessons from AFML Chapter 11:**

1. **Selection bias is real.** Testing 100 strategies, you'd expect
   a max Sharpe of ~2.5 *by chance*. Most published backtests don't
   account for this.

2. **Deflated Sharpe Ratio** corrects for:
   - Number of trials (selection bias)
   - Non-normality (fat tails, skewness)
   - Sample size

3. **PBO via CPCV** gives a direct estimate of overfitting probability.
   Low IS–OOS rank correlation → strategy selection doesn't persist OOS.

4. **For production:** before deploying any strategy, compute DSR and PBO.
   Only deploy strategies with DSR > 0.95 and PBO < 0.3.

---

*Next: [10_portfolio.ipynb](10_portfolio.ipynb) — Portfolio Construction (HRP + Meta-labeling Signals)*